# Unsloth Fine-Tuning (Colab or Kaggle)

Welcome back! In the classic notebook (`qlora_finetune.ipynb`) you fine-tuned a **0.5B**
model with the standard Hugging Face stack — that taught you the mechanics. This notebook
is the **2026-practical version**: same data, same grading, but powered by
[Unsloth](https://unsloth.ai) — a free library that trains ~2× faster with far less GPU
memory. That headroom lets us fine-tune a **6× bigger model (Llama 3.2 3B)** on the *same
free GPU*.

**Where to run this:** Google Colab (Runtime → Change runtime type → **T4 GPU**) or
Kaggle (Settings → Accelerator → **GPU T4**). Not sure which? Read
[`docs/free-gpu-guide.md`](https://github.com/ArunRyzen/lora-finetune-lab/blob/main/docs/free-gpu-guide.md)
— short version: Colab for quick sessions, Kaggle for its 30 free GPU-hours per week.

**The plan (same 4 beats as the classic notebook):**
1. Install → 2. Build the dataset → 3. Grade the *un-trained* model, train, grade again → 4. Read the verdict.


## Step 1 — Install

This cell looks scarier than it is: on Colab, specific library versions must match the
pre-installed PyTorch, so Unsloth's official recipe pins them one by one. On Kaggle (or
any other machine) the plain `pip install unsloth` branch runs instead. **Copy it as-is —
you never need to understand the pins.** We also install this repo (for the dataset +
grading helpers you already know).


In [ ]:
%%capture
# 1. Install Unsloth (official Colab/Kaggle recipe) + this repo's helpers.
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth   # Kaggle / other machines: the simple path
else:
    # Colab: pin versions that match the pre-installed torch (Unsloth's official recipe).
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip -q install "git+https://github.com/ArunRyzen/lora-finetune-lab.git"


In [ ]:
# Sanity check: do we actually have a GPU?
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — enable a T4 first!')


## Step 2 — Build the dataset

Identical to the classic notebook, on purpose: the *only* variable we change is the
training engine + model size, so any score difference is explained by those. Fine-tuning
data is just a list of (instruction, input, expected answer) examples; `format_text` turns
each one into a single training string. Swap in your own JSONL for a real task.


In [ ]:
# 2. Build data: synthetic instruction examples (swap in your own JSONL for real tasks).
from lora_finetune_lab.dataset import generate_synthetic, split
from lora_finetune_lab.prompts import format_text

examples = generate_synthetic(400)
train, val = split(examples, val_fraction=0.1)
print(len(train), 'train /', len(val), 'val')
print(format_text(train[0]))


## Step 3a — Load the model with Unsloth

Two calls:
- `FastLanguageModel.from_pretrained` downloads **Llama 3.2 3B Instruct** in **4-bit**
  (each weight squeezed into 4 bits instead of 16 — that's the "Q" in QLoRA and why 3B
  fits in a free T4's memory).
- `get_peft_model` freezes all 3B weights and bolts on tiny trainable **LoRA adapters**
  (the add-on layers — we train only those, ~1% of the size).


In [ ]:
# 3a. Load Llama 3.2 3B in 4-bit and attach LoRA adapters (Unsloth's official settings).
from unsloth import FastLanguageModel

max_seq_length = 1024   # longest training text we'll feed in (in tokens)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=max_seq_length,
    load_in_4bit=True,          # 4-bit = the memory trick that fits 3B on a free T4
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # adapter size (same knob as lora_r in our TrainingConfig)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',   # Unsloth's extra memory saver
    random_state=3407,
)


## Step 3b — Grade the model BEFORE training

Neat fact: fresh LoRA adapters start as **all zeros**, so right now the model answers
*exactly* like the untouched base model. Grading it **before** training gives us an honest
"base" score — measured on the very same object we're about to train, so the comparison
can't cheat.


In [ ]:
# 3b. Baseline score: adapters are still zeros, so this IS the base model.
from lora_finetune_lab.evaluation import accuracy

class UnslothModel:
    """Tiny wrapper so our repo's grader can talk to the Unsloth model."""
    def __init__(self, model, tokenizer):
        self.model, self.tok = model, tokenizer
    def generate(self, prompt: str) -> str:
        ids = self.tok(prompt, return_tensors='pt').to(self.model.device)
        out = self.model.generate(**ids, max_new_tokens=16, do_sample=False, use_cache=True)
        return self.tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

FastLanguageModel.for_inference(model)          # switch to fast answering mode
base_score = accuracy(UnslothModel(model, tokenizer), val)
print('BASE score (before training):', base_score)


## Step 3c — Train

The trainer shows one line per step; watch the **loss** number fall — that's learning.
`max_steps=60` keeps this to a few minutes; raise it (or switch to epochs) for real tasks.
`adamw_8bit` is a memory-saving optimizer; batch 2 × accumulation 4 = an effective batch
of 8 without T4-breaking memory use.


In [ ]:
# 3c. Train the adapters (only ~1% of weights move; the 3B base stays frozen).
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

FastLanguageModel.for_training(model)           # back to training mode
train_texts = [format_text(ex) + tokenizer.eos_token for ex in train]  # EOS = 'stop here'
dataset = Dataset.from_dict({'text': train_texts})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        optim='adamw_8bit',
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs',
        report_to='none',
    ),
)
trainer.train()


## Step 4 — The verdict: same model, after training

Same grader, same held-out validation questions, same object — the only thing that changed
is 60 steps of training on the adapters.


In [ ]:
# 4. Grade again and compare.
FastLanguageModel.for_inference(model)
tuned_score = accuracy(UnslothModel(model, tokenizer), val)
print('BASE  :', base_score)
print('TUNED :', tuned_score)

# Keep your work: the adapter is tiny (megabytes), the 3B base is not saved (not needed).
model.save_pretrained('outputs/unsloth_adapter')
tokenizer.save_pretrained('outputs/unsloth_adapter')
print('Adapter saved to outputs/unsloth_adapter')


## Read the result, then decide

TUNED should clearly beat BASE — on a *3B* model this time, trained free. Before you close
the tab:

- **Download the adapter** (it's small): Colab → Files sidebar → `outputs/unsloth_adapter`;
  Kaggle → it appears under the notebook's Output tab after "Save Version".
- **Same question as always:** did you need fine-tuning for this, or would RAG / a better
  prompt have done it? Revisit
  [`docs/when-to-finetune.md`](https://github.com/ArunRyzen/lora-finetune-lab/blob/main/docs/when-to-finetune.md).
- **For your resume:** "Fine-tuned Llama 3.2 3B with QLoRA + Unsloth on a free T4;
  measured base-vs-tuned accuracy on a held-out set" — and you can now explain every word
  of that sentence.
